In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import time
import json
import random
from tqdm.auto import tqdm


In [ ]:
# Доли пропущенных признаков для экспериментов
MISSING_RATES = [0.0, 0.2, 0.5, 0.9]

# Настройка промпта для Car

prompt_config = {
    "task": "Predict car evaluation (unacceptable, acceptable, good, very good)",
    "labels": ["unacceptable", "acceptable", "good", "very good"],
    "entity": "Car",
    "question": "What is the evaluation of this car?"
}

openml_id = 40975

base_output_dir = "/content/drive/MyDrive/finetuned_qwen_missing_Car"

# 1. Загрузка данных

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_dataset(openml_id=1464, prompt_config=None):
    dataset = fetch_openml(data_id=openml_id, as_frame=True, parser='auto')
    X = dataset.data
    y = dataset.target

    df = X.copy()
    feature_names = X.columns.to_list()

    target_name = y.name
    df[target_name] = y

    # Преобразование целевой переменной в бинарный формат
    if y.dtype == 'object' or y.dtype.name == 'category':
        le = LabelEncoder()
        df[target_name] = le.fit_transform(df[target_name])
        class_names = prompt_config['labels']
    else:
        class_names = sorted(df[target_name].unique().tolist())

    return df, feature_names, target_name, class_names


def split_dataset(df, target_name, test_size=0.2, val_size=0.25, seed=42):

    # Разделение на train/val/test (60/20/20)
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df[target_name]
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_size,
        random_state=seed,
        stratify=train_val_df[target_name]
    )

    return train_df, val_df, test_df

df, feature_names, target_name, class_names = load_dataset(openml_id, prompt_config)
train_df, val_df, test_df = split_dataset(df, target_name)

df.head(5)

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,2
1,vhigh,vhigh,2,2,small,med,2
2,vhigh,vhigh,2,2,small,high,2
3,vhigh,vhigh,2,2,med,low,2
4,vhigh,vhigh,2,2,med,med,2


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1728 entries, 0 to 1727
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   buying    1728 non-null   category
 1   maint     1728 non-null   category
 2   doors     1728 non-null   category
 3   persons   1728 non-null   category
 4   lug_boot  1728 non-null   category
 5   safety    1728 non-null   category
 6   class     1728 non-null   int64   
dtypes: category(6), int64(1)
memory usage: 24.7 KB


In [ ]:
df['class'].value_counts()

,count
class,
2,1210
0,384
1,69
3,65


# 2. Вспомогательные функции

In [ ]:
def row_to_text_template_with_missing(row, feature_names, target_name, missing_rate=0.0, seed=None):

    rng = np.random.default_rng(seed)

    # Определяем, какие признаки пропустить
    n_drop = int(len(feature_names) * missing_rate)
    dropped = set(rng.choice(feature_names, size=n_drop, replace=False)) if n_drop > 0 else set()

    template_parts = []

    for feature in feature_names:
        if feature in dropped:
            continue  # пропускаем признак

        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts)

    return text

# Тест
test_row = train_df.iloc[0]
display(train_df.head(1))
for rate in MISSING_RATES:
    text_output = row_to_text_template_with_missing(
        row=test_row,
        feature_names=feature_names,
        target_name=target_name,
        missing_rate=rate
    )

    print(f"Missing Rate: {rate * 100:.0f}%")
    print(text_output)

,buying,maint,doors,persons,lug_boot,safety,class
244,vhigh,med,3,2,small,med,2


Missing Rate: 0%
The category of buying is vhigh. The category of maint is med. The category of doors is 3. The category of persons is 2. The category of lug_boot is small. The category of safety is med.
Missing Rate: 20%
The category of buying is vhigh. The category of maint is med. The category of doors is 3. The category of lug_boot is small. The category of safety is med.
Missing Rate: 50%
The category of maint is med. The category of lug_boot is small. The category of safety is med.
Missing Rate: 90%
The category of buying is vhigh.


In [ ]:
def parse_prediction(response, prompt_config):
    """Парсинг ответа модели в номер класса"""
    response = response.lower().strip()
    response = response.rstrip('.,!? ')

    # Проверка каждого класса
    for i, class_name in enumerate(prompt_config['labels']):
        class_lower = class_name.lower()

        # Точное совпадение
        if response == class_lower:
            return i

        # Начинается с имени класса
        if response.startswith(class_lower):
            return i

        # Содержит как отдельное слово
        if class_lower in response.split():
            return i

    # Не удалось распознать - возвращаем первый класс
    print(f"Warning: Could not parse '{response}' (expected one of {prompt_config['labels']})")
    return 0

response = "good"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

response = "no"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

Response: 'good'
Prediction: 2
Response: 'no'
Prediction: 0


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Вычисление метрик качества
def compute_metrics(y_true, y_pred, y_prob=None):
    return {
        "ROC-AUC": roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro" if y_prob is not None else y_pred),
        "F1":       f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall":   recall_score(y_true, y_pred, average="macro", zero_division=0),
    }

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot, multi_class="ovr", average="macro")
            else:
                auc = 0.0

            f1 = f1_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)

            acc = accuracy_score(y_true_boot, y_pred_boot)
            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            # Пропускание сэмплов, где не представлены все классы
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

#Загрузка модели Qwen 3.0 4B Instruct
model_name = "Qwen/Qwen3-4B-Instruct-2507"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Загрузка токенизаторов
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Загрузка модель с 16-битной квантизацией и автоматическим распределением по устройствам
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device, # Explicitly force all model layers to the detected device (GPU if available)
    attn_implementation="sdpa"
)

if torch.cuda.is_available():
    print(f"Использовано памяти: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Использовано памяти: 8.04 GB


In [ ]:
def create_prompt_missing(row, feature_names, target_name, prompt_config, tokenizer, missing_rate=0.0, seed=None):

    labels_str = "', '".join(prompt_config['labels'])

    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"Answer with only one word from: '{labels_str}'."
    )

    input_text = row_to_text_template_with_missing(row, feature_names, target_name, missing_rate, seed)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",
         "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True # добавление role assistant
    )

In [ ]:
import torch.nn.functional as F

def predict_single_with_prob(prompt, prompt_config, model, tokenizer, device, max_new_tokens=5):

    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Декодирование текста
    generated_ids = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Извлечение вероятностей для меток из prompt_config
    first_token_logits = outputs.scores[0][0]

    labels_ids = []
    for labels_name in prompt_config['labels']:
        labels_id = tokenizer.encode(labels_name, add_special_tokens=False)[0]
        labels_ids.append(labels_id)


    labels_logits = torch.stack([first_token_logits[cid] for cid in labels_ids])
    probs = F.softmax(labels_logits, dim=0)

    return response, probs.cpu().numpy()

# Тест
test_prompt = create_prompt_missing(test_df.iloc[0], feature_names, target_name, prompt_config, tokenizer)
response, probs = predict_single_with_prob(test_prompt, prompt_config, base_model, tokenizer, device)
print(f"Response: '{response}'")
print(f"Probabilities: {dict(zip(prompt_config['labels'], probs))}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Response: 'good'
Probabilities: {'unacceptable': np.float32(8.8771373e-13), 'acceptable': np.float32(0.00012339411), 'good': np.float32(0.9998729), 'very good': np.float32(3.7261796e-06)}


# 3. Fine-tuning с LoRA

## Подготовка данных для Fine-tuning

In [ ]:
def balance_dataset(train_df, target_name, prompt_config, seed=42):

    labels_counts = train_df[target_name].value_counts()
    print(f"\nДо балансировки:")
    for cls, count in labels_counts.items():
        print(f"  Класс {prompt_config['labels'][cls]}: {count}")


    # Oversample до максимального класса
    max_count = labels_counts.max()
    balanced_dfs = []

    for cls in labels_counts.index:
        df_labels = train_df[train_df[target_name] == cls]
        if len(df_labels) < max_count:
            df_upsampled = resample(
                df_labels,
                replace=True,
                n_samples=max_count,
                random_state=seed
            )
            balanced_dfs.append(df_upsampled)
        else:
            balanced_dfs.append(df_labels)

    train_df_balanced = pd.concat(balanced_dfs)
    train_df_balanced = train_df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

    print(f"\nПосле балансировки:")
    print(train_df_balanced[target_name].value_counts())

    return train_df_balanced

train_df_balanced = balance_dataset(train_df, target_name, prompt_config)


До балансировки:
  Класс good: 726
  Класс unacceptable: 230
  Класс acceptable: 41
  Класс very good: 39

После балансировки:
class
2    726
0    726
1    726
3    726
Name: count, dtype: int64


In [ ]:
# Создание текстов для обучения

def create_training_dataset(df, feature_names, target_name, prompt_config, tokenizer, missing_rate=0.0):
    labels_str = "', '".join(prompt_config['labels'])
    system_prompt = (
        f"You are a classifier. {prompt_config['task']}. "
        f"Answer with only one word from: '{labels_str}'."
    )
    texts = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Dataset (missing={missing_rate:.0%})"):
        input_text = row_to_text_template_with_missing(row, feature_names, target_name)
        target = prompt_config['labels'][int(row[target_name])]

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
            {"role": "assistant", "content": target}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return texts


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

def finetune_model(missing_rate, train_df_balanced, feature_names, target_name,
                   prompt_config, tokenizer, base_model, device,
                   num_epochs=3, batch_size=16):
    """
    Полный цикл fine-tuning для заданного missing_rate.
    Возвращает путь к лучшему checkpoint.
    """
    output_dir = f"{base_output_dir}_missing{int(missing_rate * 100):03d}"
    print(f"  Fine-tuning: missing_rate = {missing_rate:.0%}")
    print(f"  Output dir : {output_dir}")

    # Датасет
    train_texts = create_training_dataset(
        train_df_balanced, feature_names, target_name,
        prompt_config, tokenizer, missing_rate=missing_rate)
    train_dataset = Dataset.from_dict({"text": train_texts})

    def tokenize_fn(examples):
        return tokenizer(examples["text"], truncation=True, max_length=512, padding=False)

    tokenized_dataset = train_dataset.map(tokenize_fn, batched=True,
                                          remove_columns=train_dataset.column_names)

    # LoRA
    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model_lora = get_peft_model(base_model, lora_config)
    model_lora.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        optim="adamw_torch_fused",
        warmup_steps=50,
        max_grad_norm=1.0,
        weight_decay=0.01,
        report_to="none",
        dataloader_num_workers=4,
        group_by_length=True,
    )

    trainer = Trainer(
        model=model_lora,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    )

    t0 = time.time()
    trainer.train()
    print(f"Обучение завершено за {(time.time()-t0)/60:.1f} мин")

    trainer.save_model(output_dir)

    # Возвращаем базовую модель без LoRA весов (важно для следующего эксперимента)
    model_lora = model_lora.unload()
    torch.cuda.empty_cache()

    return output_dir

Mounted at /content/drive


In [ ]:
import glob
import re
from peft import PeftModel

def evaluate_checkpoints_on_val(output_dir, val_df, feature_names, target_name,
                                 prompt_config, tokenizer, base_model, device,
                                 eval_missing_rate=0.0):

    checkpoints = glob.glob(f"{output_dir}/checkpoint-*")
    checkpoints = sorted(
        glob.glob(f"{output_dir}/checkpoint-*"),
        key=lambda x: int(re.search(r'checkpoint-(\d+)', x).group(1))
    )
    print(f"Найдено {len(checkpoints)} checkpoints в {output_dir}")

    best_auc, best_checkpoint = 0.0, None
    for cp in checkpoints:
        model_eval = PeftModel.from_pretrained(base_model, cp).eval()

        y_true, y_pred, y_prob = [], [], []
        for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc=f"Val eval {cp.split('/')[-1]}"):
            prompt = create_prompt_missing(row, feature_names, target_name, prompt_config, tokenizer)
            response, probs = predict_single_with_prob(prompt, prompt_config, model_eval, tokenizer, device)
            prediction = parse_prediction(response, prompt_config)

            y_true.append(row[target_name])
            y_pred.append(prediction)
            y_prob.append(probs)

        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro" if y_prob is not None else y_pred)
        print(f"  {cp.split('/')[-1]}  ROC-AUC={auc:.4f}")

        if auc > best_auc:
            best_auc = auc
            best_checkpoint = cp

        del model_eval
        torch.cuda.empty_cache()

    print(f"Лучший checkpoint: {best_checkpoint}  (ROC-AUC={best_auc:.4f})")
    return best_checkpoint

In [ ]:
def evaluate_on_test(best_checkpoint, test_df, feature_names, target_name,
                     prompt_config, tokenizer, base_model, device,
                     eval_missing_rate=0.0):
    model_finetuned = PeftModel.from_pretrained(base_model, best_checkpoint).eval()

    y_true, y_pred, y_prob = [], [], []
    t0 = time.time()

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test eval"):
        prompt = create_prompt_missing(
            row, feature_names, target_name, prompt_config, tokenizer,
            missing_rate=eval_missing_rate)
        response, prob = predict_single_with_prob(
            prompt, prompt_config, model_finetuned, tokenizer, device)
        y_true.append(row[target_name])
        y_pred.append(parse_prediction(response, prompt_config))
        y_prob.append(prob)

    elapsed = time.time() - t0
    metrics = compute_metrics(np.array(y_true), np.array(y_pred), np.array(y_prob))
    boot = bootstrap_metrics(np.array(y_true), np.array(y_pred), np.array(y_prob), n_iter=1000)

    print("Метрики:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    print("Bootstrap (mean±std):")
    for k, v in boot.items():
        print(f"  {k}: {v}")

    del model_finetuned
    torch.cuda.empty_cache()

    return {
        "metrics": metrics,
        "bootstrap": boot,
        "time_total": elapsed,
        "time_per_sample": elapsed / len(y_true),
        "best_checkpoint": best_checkpoint,
        "eval_missing_rate": eval_missing_rate,
    }

In [ ]:
all_results = {}

start_time = time.time()

for missing_rate in MISSING_RATES:
    tag = f"missing_{int(missing_rate * 100):03d}pct"
    print(f"Эксперимент: missing_rate = {missing_rate:.0%}")

    # 7.1 Fine-tuning
    output_dir = finetune_model(
        missing_rate=missing_rate,
        train_df_balanced=train_df_balanced,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        base_model=base_model,
        device=device,
        num_epochs=10,
        batch_size=16,
    )

    # 7.2 Выбор лучшего checkpoint по val
    best_cp = evaluate_checkpoints_on_val(
        output_dir=output_dir,
        val_df=val_df,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        base_model=base_model,
        device=device,
        eval_missing_rate=missing_rate,
    )

    # 7.3 Оценка на test
    result = evaluate_on_test(
        best_checkpoint=best_cp,
        test_df=test_df,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        base_model=base_model,
        device=device,
        eval_missing_rate=missing_rate,
    )

    all_results[tag] = result

print(f"Эксперимент завершен за {(time.time()-start_time)/60:.1f} мин")

Эксперимент: missing_rate = 0%
  Fine-tuning: missing_rate = 0%
  Output dir : /content/drive/MyDrive/finetuned_qwen_missing_Car_missing000


Dataset (missing=0%):   0%|          | 0/2904 [00:00<?, ?it/s]

Map:   0%|          | 0/2904 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


...

Val eval checkpoint-910:   0%|          | 0/346 [00:00<?, ?it/s]

  checkpoint-910  ROC-AUC=0.9999
Лучший checkpoint: /content/drive/MyDrive/finetuned_qwen_missing_Car_missing090/checkpoint-546  (ROC-AUC=1.0000)


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Test eval:   0%|          | 0/346 [00:00<?, ?it/s]

Метрики:
  ROC-AUC: 0.6613
  F1: 0.2634
  Accuracy: 0.5260
  Precision: 0.2837
  Recall: 0.3053
Bootstrap (mean±std):
  ROC-AUC: 0.6629±0.0266
  F1: 0.2631±0.0195
  Accuracy: 0.5264±0.0269
  Precision: 0.2841±0.0208
  Recall: 0.3083±0.0375
Эксперимент завершен за 114.8 мин


In [ ]:
all_results

{'missing_000pct': {'metrics': {'ROC-AUC': np.float64(1.0),
   'F1': 0.7332060880042482,
   'Accuracy': 0.9595375722543352,
   'Precision': 0.7205882352941176,
   'Recall': 0.7467532467532467},
  'bootstrap': {'ROC-AUC': '1.0000±0.0000',
   'F1': '0.7329±0.0113',
   'Accuracy': '0.9598±0.0106',
   'Precision': '0.7207±0.0174',
   'Recall': '0.7467±0.0033'},
  'time_total': 88.11212229728699,
  'time_per_sample': 0.25465931299793926,
  'best_checkpoint': '/content/drive/MyDrive/finetuned_qwen_missing_Car_missing000/checkpoint-546',
  'eval_missing_rate': 0.0},
 'missing_020pct': {'metrics': {'ROC-AUC': np.float64(0.9487573483596758),
   'F1': 0.5660455486542443,
   'Accuracy': 0.838150289017341,
   'Precision': 0.5328245406046236,
   'Recall': 0.6130460448642266},
  'bootstrap': {'ROC-AUC': '0.9494±0.0119',
   'F1': '0.5650±0.0285',
   'Accuracy': '0.8391±0.0196',
   'Precision': '0.5339±0.0299',
   'Recall': '0.6141±0.0309'},
  'time_total': 89.35413551330566,
  'time_per_sample': 0.25

In [ ]:
print("\nBootstrap (mean±std):")
for tag, res in all_results.items():
    rate = tag.replace("missing_", "").replace("pct", "%").lstrip("0") or "0%"
    print(f"\n  missing={rate}")
    for k, v in res["bootstrap"].items():
        print(f"    {k}: {v}")


Bootstrap (mean±std):

  missing=%
    ROC-AUC: 1.0000±0.0000
    F1: 0.7329±0.0113
    Accuracy: 0.9598±0.0106
    Precision: 0.7207±0.0174
    Recall: 0.7467±0.0033

  missing=20%
    ROC-AUC: 0.9494±0.0119
    F1: 0.5650±0.0285
    Accuracy: 0.8391±0.0196
    Precision: 0.5339±0.0299
    Recall: 0.6141±0.0309

  missing=50%
    ROC-AUC: 0.8002±0.0218
    F1: 0.3497±0.0314
    Accuracy: 0.6486±0.0253
    Precision: 0.3419±0.0318
    Recall: 0.3629±0.0350

  missing=90%
    ROC-AUC: 0.6629±0.0266
    F1: 0.2631±0.0195
    Accuracy: 0.5264±0.0269
    Precision: 0.2841±0.0208
    Recall: 0.3083±0.0375


ROC-AUC (1.00): Идеальный показатель производительности при полном наборе данных. Это свидетельствует о том, что модель Qwen полностью усвоила логику оценки автомобилей. При 20% пропусков показатель остается очень высоким (0.95), что говорит о колоссальной избыточности ключевых признаков.

Accuracy (0.96): Исключительная общая точность. Модель практически безошибочно распределяет автомобили по категориям. При увеличении пропусков до 50% точность снижается до 0.65, что указывает на высокую чувствительность данной задачи к потере конкретных технических характеристик.

Recall (0.75): Хороший показатель полноты для многоклассовой задачи. Модель уверенно идентифицирует целевые классы. Однако наблюдается резкое падение до 0.31 при 90% пропусков, так как без ключевых данных (например, цена или вместимость) классификация становится близкой к случайной.

Precision (0.72): Точность предсказаний сбалансирована с полнотой. В отличие от бинарных задач (Bank, Blood), здесь модель менее «агрессивна», что минимизирует количество ложных классификаций. При экстремальных пропусках (90%) точность удерживается на уровне 0.28, что всё еще выше статистического порога для четырех классов.

Время эксперимента: ~ 1 ч 54 мин.

Использовано оперативная памяти графического процесса: 26.8/80GB.

Графический процессор A100 80GB.